# 01 — Data Pipeline

## Task 1.2 — Open-Meteo weather fetch

Fetches hourly `temperature_2m` and `shortwave_radiation` (GHI) from the
Open-Meteo historical archive for a given location and date range.

In [1]:
import ssl
import requests
import pandas as pd
from requests.adapters import HTTPAdapter

In [2]:
class _WinCertAdapter(HTTPAdapter):
    """
    Mounts Windows system CA store so requests works behind corporate proxies.
    """
    def init_poolmanager(self, *args, **kwargs):
        ctx = ssl.create_default_context()
        ctx.load_default_certs(ssl.Purpose.SERVER_AUTH)
        kwargs["ssl_context"] = ctx
        super().init_poolmanager(*args, **kwargs)

_session = requests.Session()
_session.mount("https://", _WinCertAdapter())


def fetch_weather(lat, lon, start_date, end_date, timezone) -> pd.DataFrame:
    """
    Returns hourly UTC-aware DataFrame indexed by timestamp,
       with columns: temperature_2m, shortwave_radiation (Global Horizontal Irradiance, W/m²).

    Data is fetched and stored in UTC regardless of the timezone arg.
    Convert to local time only for plotting — solar physics works in UTC + longitude.
    """
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,shortwave_radiation",
        "timezone": "UTC",  # always fetch UTC — avoids DST NonExistentTime/AmbiguousTime errors
    }
    resp = _session.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params=params,
        timeout=30,
    )
    resp.raise_for_status()

    hourly = resp.json()["hourly"]
    df = pd.DataFrame({
        "temperature_2m":      hourly["temperature_2m"],
        # shortwave_radiation == GHI (Global Horizontal Irradiance, W/m²)
        # pvlib will decompose this into DNI + DHI components
        "shortwave_radiation": hourly["shortwave_radiation"],
    }, index=pd.to_datetime(hourly["time"], utc=True))  # tz-aware UTC in one step, no tz_localize
    df.index.name = "timestamp"
    return df

In [3]:
# Austin, TX — single source of truth; Week 2 pvlib uses these same coords
AUSTIN_LAT = 30.2672
AUSTIN_LON  = -97.7431
LOCAL_TZ    = "America/Chicago"

In [4]:
# Smoke test — full year to exercise DST transitions
df_weather = fetch_weather(
    lat=AUSTIN_LAT,
    lon=AUSTIN_LON,
    start_date="2018-01-01",
    end_date="2018-12-31",
    timezone=LOCAL_TZ,
)

assert df_weather.index.tz is not None,               "Index must be tz-aware"
assert not df_weather.index.has_duplicates,            "Duplicate timestamps (DST fold?)"
assert df_weather.index.is_monotonic_increasing,       "Index not sorted"

# Open-Meteo occasionally returns null for shortwave_radiation at range boundaries
n_nans = df_weather.isna().sum().sum()
if n_nans:
    print(f"WARNING: {n_nans} NaNs present — investigate")

print(f"Rows: {len(df_weather)} (expect 8760 for a non-leap year)")
print(f"Index dtype: {df_weather.index.dtype}")
print(df_weather.head())

Rows: 8760 (expect 8760 for a non-leap year)
Index dtype: datetime64[us, UTC]
                           temperature_2m  shortwave_radiation
timestamp                                                     
2018-01-01 00:00:00+00:00             0.0                  8.0
2018-01-01 01:00:00+00:00            -0.8                  0.0
2018-01-01 02:00:00+00:00            -1.0                  0.0
2018-01-01 03:00:00+00:00            -1.2                  0.0
2018-01-01 04:00:00+00:00            -1.2                  0.0


## Task 1.3 — Merge skeleton (built and proven against a mock)

In [5]:
import numpy as np

# ---------------------------------------------------------------------------
# Mock weather: 48 h of hourly UTC data (2018-01-01 – 2018-01-02)
# ---------------------------------------------------------------------------
_mock_weather_idx = pd.date_range("2018-01-01", periods=48, freq="h", tz="UTC")
mock_weather_df = pd.DataFrame({
    "temperature_2m":      np.linspace(5, 15, 48),
    "shortwave_radiation": np.clip(np.sin(np.linspace(0, 4 * np.pi, 48)) * 400, 0, None),
}, index=_mock_weather_idx)
mock_weather_df.index.name = "timestamp"

# ---------------------------------------------------------------------------
# Mock energy: same window but with four deliberate defects:
#   (a) missing hour        — 2018-01-01 15:00 UTC dropped entirely
#   (b) NaN load value      — household_load_kwh at 2018-01-01 20:00 set to NaN
#   (c) out-of-range rows   — two rows outside the weather window (before and after)
#   (d) NaN PV value        — actual_pv_yield_kwh at 2018-01-01 10:00 set to NaN
#                             tests that the "PV stays NaN" policy applies to genuine
#                             measured-then-dropped readings, not just reindex-created gaps
# ---------------------------------------------------------------------------
_energy_idx = pd.date_range("2018-01-01", periods=48, freq="h", tz="UTC")
mock_energy_df = pd.DataFrame({
    "household_load_kwh":  np.random.default_rng(42).uniform(0.3, 1.2, 48),
    "actual_pv_yield_kwh": np.clip(np.sin(np.linspace(0, 4 * np.pi, 48)) * 2, 0, None),
}, index=_energy_idx)
mock_energy_df.index.name = "timestamp"

# (a) drop one hour
mock_energy_df = mock_energy_df.drop(pd.Timestamp("2018-01-01 15:00", tz="UTC"))

# (b) inject a NaN into load
mock_energy_df.loc[pd.Timestamp("2018-01-01 20:00", tz="UTC"), "household_load_kwh"] = np.nan

# (c) append two out-of-range rows
_extra = pd.DataFrame({
    "household_load_kwh":  [0.9, 0.8],
    "actual_pv_yield_kwh": [0.0, 0.0],
}, index=pd.to_datetime(["2017-12-31 23:00", "2018-01-03 00:00"], utc=True))
_extra.index.name = "timestamp"
mock_energy_df = pd.concat([mock_energy_df, _extra]).sort_index()

# (d) inject a NaN into PV at a row that exists (not a dropped/reindex gap)
mock_energy_df.loc[pd.Timestamp("2018-01-01 10:00", tz="UTC"), "actual_pv_yield_kwh"] = np.nan

print(f"mock_weather rows : {len(mock_weather_df)}")
print(f"mock_energy rows  : {len(mock_energy_df)}")  # print actual count, don't trust arithmetic
print(f"Hour 15:00 missing from energy: {pd.Timestamp('2018-01-01 15:00', tz='UTC') not in mock_energy_df.index}")

mock_weather rows : 48
mock_energy rows  : 49
Hour 15:00 missing from energy: True


In [6]:
def build_unified_frame(weather_df: pd.DataFrame, energy_df: pd.DataFrame) -> pd.DataFrame:
    """
    Align energy data onto weather data on a complete hourly UTC index.
    Returns one frame: timestamp index +
      [temperature_2m, shortwave_radiation, household_load_kwh, actual_pv_yield_kwh]

    Join direction: weather range is canonical.
      - Week 2 physics prediction requires a weather row for every output timestamp;
        we cannot produce a solar estimate without it, so the weather window drives the index.
      - Energy rows outside that window are irrelevant and dropped by reindex.
      - Week 3 training handles residual NaNs by dropping those rows at fit-time.

    Missing energy hours:
      - household_load_kwh  → interpolate(method="time", limit=2, limit_direction="both").
          limit=2: gaps of 1–2 hours are estimable from neighbors; longer gaps (sensor outage)
          are left as NaN and dropped at training time rather than fabricated.
          limit_direction="both": handles edge NaNs where one neighbor is missing (start/end
          of window misalignment with real Pecan Street data).
      - actual_pv_yield_kwh → left as NaN in all cases. Solar yield depends on actual cloud
          cover; interpolating it would fabricate training signal.
    """
    # 1. Canonical index: every hour in the weather window, no gaps
    canonical = pd.date_range(
        start=weather_df.index.min(),
        end=weather_df.index.max(),
        freq="h",
        tz="UTC",
    )

    # 2. Reindex both sources — out-of-range energy rows dropped, gaps become NaN
    weather_aligned = weather_df.reindex(canonical)
    energy_aligned  = energy_df.reindex(canonical)

    # 3. Impute load only; PV intentionally stays NaN
    energy_aligned["household_load_kwh"] = energy_aligned["household_load_kwh"].interpolate(
        method="time",
        limit=2,               # gaps > 2 h left as NaN — don't invent a half-day load curve
        limit_direction="both", # fill edge NaNs from the available neighbor
    )

    # 4. Combine and name index
    result = pd.concat([weather_aligned, energy_aligned], axis=1)
    result.index.name = "timestamp"
    return result

In [7]:
result = build_unified_frame(mock_weather_df, mock_energy_df)

# Index integrity
assert result.index.tz is not None,               "Index must be tz-aware"
assert not result.index.has_duplicates,            "Duplicate timestamps"
assert result.index.is_monotonic_increasing,       "Index not sorted"
assert len(result) == len(mock_weather_df),        "Row count must match weather window"

# (c) Out-of-range energy rows must be gone
assert pd.Timestamp("2017-12-31 23:00", tz="UTC") not in result.index, "Pre-range row leaked in"
assert pd.Timestamp("2018-01-03 00:00", tz="UTC") not in result.index, "Post-range row leaked in"

# (a) Missing hour restored; load interpolated, PV stays NaN (reindex-created gap)
missing_ts = pd.Timestamp("2018-01-01 15:00", tz="UTC")
assert missing_ts in result.index,                                        "Missing hour not restored"
assert pd.notna(result.loc[missing_ts, "household_load_kwh"]),            "load not interpolated at missing hour"
assert pd.isna(result.loc[missing_ts, "actual_pv_yield_kwh"]),            "PV should stay NaN at missing hour"

# (b) Explicit NaN in load interpolated
nan_load_ts = pd.Timestamp("2018-01-01 20:00", tz="UTC")
assert pd.notna(result.loc[nan_load_ts, "household_load_kwh"]),           "load NaN not interpolated"

# (d) Explicit NaN in PV stays NaN — tests the policy on a measured-then-nulled reading,
#     not just a reindex-created gap
nan_pv_ts = pd.Timestamp("2018-01-01 10:00", tz="UTC")
assert pd.isna(result.loc[nan_pv_ts, "actual_pv_yield_kwh"]),             "PV NaN must survive (no interpolation)"

print(result.info())
print(f"\nNaNs per column:\n{result.isna().sum()}")
# expect: load=0, pv=2 (hour 15:00 reindex gap + hour 10:00 explicit NaN)

<class 'pandas.DataFrame'>
DatetimeIndex: 48 entries, 2018-01-01 00:00:00+00:00 to 2018-01-02 23:00:00+00:00
Freq: h
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   temperature_2m       48 non-null     float64
 1   shortwave_radiation  48 non-null     float64
 2   household_load_kwh   48 non-null     float64
 3   actual_pv_yield_kwh  46 non-null     float64
dtypes: float64(4)
memory usage: 2.9 KB
None

NaNs per column:
temperature_2m         0
shortwave_radiation    0
household_load_kwh     0
actual_pv_yield_kwh    2
dtype: int64


## Task 1.4 — Pecan Street loader: power → hourly energy

### Column audit (done on the actual Pecan Street schema before writing any code)

Pecan Street residential exports contain:

| column | meaning | units |
|--------|---------|-------|
| `use`   | **gross household consumption** — total power drawn by the home | kW |
| `solar` | PV generation | kW (positive = generating) |
| `grid`  | net grid import/export = `use − solar` | kW (negative = exporting) |

`grid` is **not** household load — it is net consumption after the solar offset.  
Using `grid` as load would under-report consumption during sunny hours and produce nonsense correlations with irradiance in Week 3.  
→ Use `use` for `household_load_kwh`, `solar` for `actual_pv_yield_kwh`.

### kW → kWh decision

`mean(kW over one hour) × 1 h = kWh`  
For evenly-spaced samples this equals the trapezoidal integral.  
`.sum()` on 60 one-minute kW readings returns `Σ kW`, dimensionally `kW·samples`, not `kWh` — it is **60× too large**.  
If the native sampling is irregular (Pecan Street 15-min data has gaps), the simple mean is still valid as long as the gaps are small relative to the hour; large gaps produce NaN via `min_count` guard (see function).

### Sign convention

`solar` in Pecan Street is stored as positive generation.  No sign flip needed.  
`grid` is ignored entirely.

In [8]:
def _hourly_energy(power_kw: pd.Series, min_valid: int) -> pd.Series:
    """mean kW per hour = kWh; NaN if fewer than min_valid sub-hour samples present.

    Uses count-then-mask rather than resample().mean(min_count=...) because
    Resampler.mean() does not accept min_count — only .sum()/.prod() do.
    """
    g = power_kw.resample("1h")
    return g.mean().where(g.count() >= min_valid)


def _infer_freq_minutes(df: pd.DataFrame) -> int:
    """Return median sampling interval in minutes."""
    diffs = df.index.to_series().diff().dropna()
    median_minutes = int(diffs.median().total_seconds() // 60)
    return median_minutes if median_minutes > 0 else 1


def load_pecan_street(csv_path, home_id) -> pd.DataFrame:
    """
    Load one Pecan Street home from the 15-min export, resample power (kW) → hourly energy (kWh).
    Returns tz-aware UTC hourly DataFrame: [household_load_kwh, actual_pv_yield_kwh]
    ready to feed straight into build_unified_frame().

    Column mapping (Pecan Street 15-min schema):
      grid + solar → use → household_load_kwh  (grid is net import; use = grid + solar)
      solar (+ solar2 if present) → actual_pv_yield_kwh (positive = generating)

    local_15min carries its own UTC offset (-06:00 / -05:00), so pd.to_datetime(..., utc=True)
    parses and converts to UTC in one step — no tz_localize/nonexistent/ambiguous needed.

    kW → kWh: _hourly_energy() takes the mean kW per hour (= kWh), min_valid=3
      (3 of 4 fifteen-min slots must be present; hours with larger gaps → NaN).
    """
    _header = pd.read_csv(csv_path, nrows=0).columns.tolist()
    _usecols = [c for c in ["dataid", "local_15min", "grid", "solar", "solar2"] if c in _header]
    df = pd.read_csv(csv_path, usecols=_usecols)
    df = df[df["dataid"] == home_id].copy()

    df.index = pd.to_datetime(df["local_15min"], utc=True)
    df.index.name = "timestamp"
    df = df.drop(columns=["dataid", "local_15min"]).sort_index()

    # Homes with two solar circuits: sum them; min_count=1 preserves NaN for genuine dropouts
    if "solar2" in df.columns:
        df["solar"] = df[["solar", "solar2"]].sum(axis=1, min_count=1)
        df = df.drop(columns=["solar2"])
    elif "solar" not in df.columns:
        df["solar"] = 0.0  # structural zero — no panels, not a sensor dropout

    # use = grid + solar is a definitional identity; no fillna on solar
    df["use"] = df["grid"] + df["solar"]

    return pd.DataFrame({
        "household_load_kwh":  _hourly_energy(df["use"],   3),
        "actual_pv_yield_kwh": _hourly_energy(df["solar"], 3),
    })

In [9]:
# ── Mock-resample proof ─────────────────────────────────────────────────────
# Three hours of 1-minute data with known values:
#   hour 0: 60 readings at 2.0 kW / 0.5 kW  → full hour  → 2.0 kWh / 0.5 kWh
#   hour 1: 60 readings at 3.0 kW / 1.0 kW  → full hour  → 3.0 kWh / 1.0 kWh
#   hour 2: 10 readings at 4.0 kW            → sparse (< 45 min_valid) → NaN

_full  = pd.date_range("2018-01-01 06:00", periods=120, freq="min", tz="UTC")  # hours 0 & 1
_spare = pd.date_range("2018-01-01 08:00", periods=10,  freq="min", tz="UTC")  # 10 of 60 in hour 2

mock_pecan = pd.DataFrame({
    "use":   ([2.0] * 60) + ([3.0] * 60),
    "solar": ([0.5] * 60) + ([1.0] * 60),
}, index=_full)

mock_sparse = pd.DataFrame({
    "use":   [4.0] * 10,
    "solar": [2.0] * 10,
}, index=_spare)

mock_all = pd.concat([mock_pecan, mock_sparse])

# Correct resample via _hourly_energy
correct = pd.DataFrame({
    "household_load_kwh":  _hourly_energy(mock_all["use"],   min_valid=45),
    "actual_pv_yield_kwh": _hourly_energy(mock_all["solar"], min_valid=45),
})

# Wrong: unbounded mean (no count guard) — sparse hour silently returns a value
wrong = pd.DataFrame({
    "load_no_guard": mock_all["use"].resample("1h").mean(),
})

# Wrong: sum — 60x too large
wrong_sum = mock_all["use"].resample("1h").sum()

print("=== Correct (_hourly_energy with min_valid=45) ===")
print(correct.to_string())
print("\n=== Wrong (mean, no count guard) ===")
print(wrong.to_string())
print("\n=== Wrong (sum, 60x too large) ===")
print(wrong_sum.to_string())

# Full-hour values are correct
assert correct["household_load_kwh"].iloc[0]  == 2.0,  "hour-0 load 2.0 kWh"
assert correct["household_load_kwh"].iloc[1]  == 3.0,  "hour-1 load 3.0 kWh"
assert correct["actual_pv_yield_kwh"].iloc[0] == 0.5,  "hour-0 solar 0.5 kWh"
assert correct["actual_pv_yield_kwh"].iloc[1] == 1.0,  "hour-1 solar 1.0 kWh"

# Sparse hour is masked to NaN, not silently returned
assert pd.isna(correct["household_load_kwh"].iloc[2]),  "sparse hour must be NaN"
assert pd.isna(correct["actual_pv_yield_kwh"].iloc[2]), "sparse solar must be NaN"

# Without the guard, mean would silently produce a value
assert pd.notna(wrong["load_no_guard"].iloc[2]),        "unguarded mean returns a value (the bug)"

# Sum is 60x too large for a full hour
assert wrong_sum.iloc[0] == 120.0,                      "sum 60x too large"

print("\nAll resample-math assertions passed.")

=== Correct (_hourly_energy with min_valid=45) ===
                           household_load_kwh  actual_pv_yield_kwh
2018-01-01 06:00:00+00:00                 2.0                  0.5
2018-01-01 07:00:00+00:00                 3.0                  1.0
2018-01-01 08:00:00+00:00                 NaN                  NaN

=== Wrong (mean, no count guard) ===
                           load_no_guard
2018-01-01 06:00:00+00:00            2.0
2018-01-01 07:00:00+00:00            3.0
2018-01-01 08:00:00+00:00            4.0

=== Wrong (sum, 60x too large) ===
2018-01-01 06:00:00+00:00    120.0
2018-01-01 07:00:00+00:00    180.0
2018-01-01 08:00:00+00:00     40.0
Freq: h

All resample-math assertions passed.


## Task 1.5 — End-to-end real-data run (Week 1 deliverable)

`load_pecan_street` → `build_unified_frame` on home 661 + 2018 weather.
This is the gate check before Week 2 physics.

In [10]:
from pathlib import Path

CSV_PATH = Path("../data/15minute_data_austin/15minute_data_austin.csv")

energy_661 = load_pecan_street(CSV_PATH, home_id=661)
unified    = build_unified_frame(df_weather, energy_661)

print("=== info() ===")
unified.info()

print("\n=== describe() ===")
print(unified.describe().to_string())

print("\n=== NaN counts ===")
print(unified.isna().sum().to_string())

# Sanity 1: PV should never be meaningfully positive at pitch-dark hours.
# Note: Open-Meteo reports shortwave_radiation=0 on heavy-overcast days too, so
# GHI==0 is not a clean "night" proxy. We check the hard floor: GHI=0 AND
# time window that is definitely night for Austin (01:00–10:00 UTC = 7pm–4am CST).
night_utc_mask = (
    (unified["shortwave_radiation"] == 0)
    & (unified.index.hour >= 1)
    & (unified.index.hour <= 10)
)
night_pv = unified.loc[night_utc_mask, "actual_pv_yield_kwh"].dropna()
assert (night_pv <= 0.05).all(), f"Solar generation during hard-night hours (max={night_pv.max():.3f} kWh)"

# Sanity 2: positive generation during bright hours
day_pv = unified.loc[unified["shortwave_radiation"] > 300, "actual_pv_yield_kwh"].dropna()
assert (day_pv > 0).any(), "No positive PV during bright hours — check solar column mapping"

# Report cloudy-day edge cases separately (not an error, just a note)
cloudy_night_pv = unified.loc[
    (unified["shortwave_radiation"] == 0) & ~night_utc_mask, "actual_pv_yield_kwh"
].dropna()
print(f"\nHard-night PV (01-10 UTC, GHI=0): {len(night_pv)} hrs, max={night_pv.max():.4f} kWh")
print(f"Edge-of-day GHI=0 PV (diffuse/overcast): {len(cloudy_night_pv)} hrs, max={cloudy_night_pv.max():.4f} kWh")
print(f"Bright-hour PV: mean={day_pv.mean():.3f} kWh, max={day_pv.max():.3f} kWh")
print("All sanity checks passed — unified frame is ready for Week 2.")

=== info() ===
<class 'pandas.DataFrame'>
DatetimeIndex: 8760 entries, 2018-01-01 00:00:00+00:00 to 2018-12-31 23:00:00+00:00
Freq: h
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   temperature_2m       8760 non-null   float64
 1   shortwave_radiation  8760 non-null   float64
 2   household_load_kwh   8724 non-null   float64
 3   actual_pv_yield_kwh  8669 non-null   float64
dtypes: float64(4)
memory usage: 342.2 KB

=== describe() ===
       temperature_2m  shortwave_radiation  household_load_kwh  actual_pv_yield_kwh
count     8760.000000          8760.000000         8724.000000          8669.000000
mean        20.354429           187.357763            1.490840             0.883481
std          9.013339           271.541170            1.189366             1.383218
min         -7.200000             0.000000            0.107250            -0.042250
25%         13.600000             0.000000          

In [11]:
from pathlib import Path

out_path = Path("../data/processed/unified_2018_661.parquet")
out_path.parent.mkdir(parents=True, exist_ok=True)
unified.to_parquet(out_path)
print(f"Saved {out_path.resolve()} ({out_path.stat().st_size // 1024} KB)")

Saved C:\Users\Frankie Lam\OneDrive\Documents\SolarOS\SolarOS\data\processed\unified_2018_661.parquet (178 KB)
